# Skolfie for teachers: web app and batch checks (roadmap step 11)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mohaaa825/Skolfie/blob/main/notebooks/app.ipynb)

Uses the models you trained with the other notebooks (by default the modern retrain in
`My Drive/skolfie/models_modern/` and `My Drive/skolfie/reports_modern/`) to

* **(a)** start the Skolfie web app with a public share link, protected by a password, so you
  (or colleagues you give the password to) can paste or upload essays from any browser;
* **(b)** check a whole Drive folder of essays (.txt, .docx, .pdf) in one go with the
  `skolfie check` command, writing a CSV of verdicts and one HTML page per essay;
* **(c)** optionally copy just the files Skolfie needs into one portable folder
  (`skolfie bundle`), e.g. to run it on a school computer.

**Use a GPU runtime** (T4) for speed; a CPU runtime works but each essay takes several
seconds. Run the cells from the top: Settings, Setup and Password first, then whichever of
(a), (b), (c) you need.

The web app needs a password because a share link is reachable by anyone who has it. Put the
password in a Colab secret named `SKOLFIE_APP_PASSWORD` (key icon in the left sidebar), or
type it when asked; it is never stored in this notebook. Results are probabilistic: never use
them as the only basis for a misconduct decision.

## 1. Settings

In [ ]:
# Retrained on modern essays (retrain_modern.ipynb); the old models stay in models/ and reports/
MODELS_DIR = "/content/drive/MyDrive/skolfie/models_modern"
REPORTS_DIR = "/content/drive/MyDrive/skolfie/reports_modern"

REPO = "mohaaa825/Skolfie"
BRANCH = "main"

# (a) web app
APP_USER = "teacher"          # login name; the password is asked for below, never written here
SHOW_FEEDBACK = False         # True: add the Feedback & Dataset panel listing every stored text
FEEDBACK_CSV = "/content/drive/MyDrive/skolfie/feedback/feedback_data.csv"

# (b) batch check: put the essays (.txt / .docx / .pdf) in this Drive folder
ESSAYS_DIR = "/content/drive/MyDrive/skolfie/essays"
RESULTS_DIR = "/content/drive/MyDrive/skolfie/results"

# (c) portable bundle
BUNDLE_DIR = "/content/drive/MyDrive/skolfie/bundle"

# (d) permanent website on Hugging Face Spaces
HF_SPACE = ""                 # e.g. "yourname/skolfie"; the site becomes yourname-skolfie.hf.space
SPACE_PRIVATE = False         # True: only your Hugging Face account can open the Space
PUBLIC_NO_LOGIN = False       # True: anyone with the link can use it, no password
SPACE_DIR = "/content/skolfie_space"

## 2. Check the GPU, mount Drive, clone and install

In [ ]:
import os
import subprocess
import sys

import torch

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Note: no GPU; checking each essay will take several seconds.")

from google.colab import drive

drive.mount("/content/drive")
for d in (os.path.dirname(FEEDBACK_CSV), RESULTS_DIR):
    os.makedirs(d, exist_ok=True)
assert os.path.exists(f"{MODELS_DIR}/stacked.joblib"), (
    f"No {MODELS_DIR}/stacked.joblib yet: run the training notebooks first "
    "(feature_baseline, transformer, stack, chunks, mixed, explain, fairness).")


def secret(name):
    try:
        from google.colab import userdata

        return userdata.get(name)
    except Exception:
        return None


def git(*args, check=True):
    """Run git; a GITHUB_TOKEN secret reaches it through a credential helper that reads it from
    the environment, so the token never appears in a URL, a command line, an error message or
    .git/config."""
    env = dict(os.environ, GIT_TERMINAL_PROMPT="0")
    auth = []
    token = secret("GITHUB_TOKEN")
    if token:
        env["SKOLFIE_GITHUB_TOKEN"] = token
        auth = ["-c", "credential.helper=", "-c",
                "credential.helper=!f() { test \"$1\" = get && echo username=x-access-token "
                "&& echo \"password=$SKOLFIE_GITHUB_TOKEN\"; }; f"]
    return subprocess.run(["git", *auth, *args], env=env, check=check)


def run(cmd, env=None):
    """Run a command and stream its output into the notebook as it happens."""
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                            bufsize=1, env=env)
    for line in proc.stdout:
        sys.stdout.write(line)
    if proc.wait() != 0:
        raise RuntimeError(f"command failed with exit code {proc.returncode}")


url = f"https://github.com/{REPO}.git"
if os.path.isdir("/content/Skolfie/.git"):
    # Clones made by older versions of this notebook kept the token in the remote URL.
    git("-C", "/content/Skolfie", "remote", "set-url", "origin", url)
    git("-C", "/content/Skolfie", "pull", "--ff-only")
else:
    git("clone", "--depth", "1", "--branch", BRANCH, url, "/content/Skolfie")

%cd /content/Skolfie
!git log --oneline -1
!pip install -q -e ".[features,app]"

## 3. Password for the web app

From the Colab secret `SKOLFIE_APP_PASSWORD` if you made one, else typed in (hidden). Use a
long password: anyone with the share link and the password can use the app.

In [ ]:
from getpass import getpass

APP_PASSWORD = secret("SKOLFIE_APP_PASSWORD") or getpass("Password for the Skolfie web app: ")
assert len(APP_PASSWORD) >= 8, "Please use a password of at least 8 characters."
print("Password set (not shown).")

## 4. (a) Start the web app

Starts Skolfie in the background and prints its public `https://….gradio.live` link once the
models are loaded (a minute or two). Log in with `APP_USER` and your password. The link works
while this notebook runs (Gradio share links expire after about a week at most); to stop the app,
set `STOP_APP = True` in the next cell and run it. Feedback sent from the app is saved to `FEEDBACK_CSV` on Drive.

The password reaches the app through an environment variable (`SKOLFIE_APP_AUTH`), not the
command line, so it does not show up in process lists or logs.

In [ ]:
import time

app_env = dict(os.environ, SKOLFIE_APP_AUTH=f"{APP_USER}:{APP_PASSWORD}",
               SKOLFIE_FEEDBACK=FEEDBACK_CSV)
cmd = [sys.executable, "-u", "-m", "skolfie.app", "--share",
       "--models-dir", MODELS_DIR, "--reports-dir", REPORTS_DIR]
if SHOW_FEEDBACK:
    cmd.append("--show-feedback")

previous = globals().get("app_proc")
if previous is not None and previous.poll() is None:  # restart: stop the old app first
    previous.terminate()
    previous.wait()
app_log = open("/content/skolfie_app.log", "w")
app_proc = subprocess.Popen(cmd, stdout=app_log, stderr=subprocess.STDOUT, env=app_env)

link = None
for _ in range(600):  # up to 10 minutes for the models to load
    time.sleep(1)
    with open("/content/skolfie_app.log", encoding="utf-8", errors="replace") as fh:
        log = fh.read()
    for line in log.splitlines():
        if "gradio.live" in line:
            link = line[line.index("https://"):].split()[0]
    if link or app_proc.poll() is not None:
        break
print(log[-3000:])
if link:
    print(f"\nSkolfie is running: {link}\nLog in as {APP_USER!r} with your password.")
else:
    print("The app did not start (see the log above).")

In [ ]:
# Stop the web app: set STOP_APP = True and run this cell. (False by default, so "Run all"
# does not stop the app it has just started.)
STOP_APP = False

running = globals().get("app_proc")
if not STOP_APP:
    print("The app keeps running; set STOP_APP = True and run this cell to stop it.")
elif running is not None and running.poll() is None:
    running.terminate()
    running.wait()
    print("Stopped.")
else:
    print("The app is not running.")


## 5. (b) Check a folder of essays

Checks every .txt, .docx and .pdf in `ESSAYS_DIR` with the `skolfie check` command (Skolfie's
command line tool) and writes `RESULTS_DIR/results.csv` (one row per essay: verdict, AI
likelihood, warnings) and `RESULTS_DIR/html/` (one page per essay with the highlighted text and
the reasons). Open the HTML pages from Drive, or see the first one below.

In [ ]:
import pandas as pd
from IPython.display import HTML, display

os.makedirs(ESSAYS_DIR, exist_ok=True)
essays = [f for _, _, files in os.walk(ESSAYS_DIR) for f in files
          if f.lower().endswith((".txt", ".md", ".docx", ".pdf"))]
if not essays:
    print(f"No essays in {ESSAYS_DIR} yet, so this step was skipped. Put .txt / .docx / .pdf "
          "files there and run this cell again.")
else:
    run(["skolfie", "check", ESSAYS_DIR,
         "--models-dir", MODELS_DIR, "--reports-dir", REPORTS_DIR,
         "--format", "csv", "--out", f"{RESULTS_DIR}/results.csv",
         "--html", f"{RESULTS_DIR}/html"])
    results = pd.read_csv(f"{RESULTS_DIR}/results.csv")
    display(results)
    pages = sorted(p for p in os.listdir(f"{RESULTS_DIR}/html") if p.endswith(".html"))
    if pages:
        with open(f"{RESULTS_DIR}/html/{pages[0]}", encoding="utf-8") as fh:
            display(HTML(fh.read()))

## 6. (c) Optional: a portable bundle

Copies the model files Skolfie needs (and the reports it reads) into `BUNDLE_DIR` with a
`bundle.json`, so the folder works anywhere: download it and start the app on another computer
with `python -m skolfie.app --bundle FOLDER --auth user:password` (after
`pip install -e ".[features,app]"` in a clone of the repository).

In [ ]:
run(["skolfie", "bundle", "--models-dir", MODELS_DIR, "--reports-dir", REPORTS_DIR,
     "--out", BUNDLE_DIR])
!du -sh "$BUNDLE_DIR"

## 7. (d) Publish a permanent website on Hugging Face Spaces

Puts the web app and your models online at `https://<user>-<name>.hf.space`, so teachers can use
it without this notebook running. "CPU basic" hardware is enough (a few seconds per essay), but Hugging Face only hosts
Gradio apps for **PRO** accounts ($9/month, [huggingface.co/pro](https://huggingface.co/pro)).

Once, before the first run:
1. Make an account at [huggingface.co](https://huggingface.co/join) and subscribe to PRO.
2. Create a token with the **Write** role at
   [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens).
3. Either add it in Colab's 🔑 (Secrets) panel as a secret named `HF_TOKEN` (switch on notebook
   access), or just paste it into the hidden box the cell shows. Never put the token in the
   notebook text: it would end up on GitHub.
4. Set `HF_SPACE` in the settings cell, e.g. `"yourname/skolfie"`.

The cell bundles the models, writes the Space (pinned to the library versions of this Colab, so
the saved models load there) and uploads it. The login is the same as for the share link
(`APP_USER` and the password from step 3). The first build on Hugging Face takes about 5–10
minutes; follow it on the Space page. The Space goes to sleep after 48 hours without visitors
and wake up in about a minute on the next visit. Run the cell again after retraining to update
the site.

In [ ]:
import sys

from skolfie.deploy.hf_space import space_page_url, space_site_url

if not HF_SPACE:
    print("HF_SPACE is empty, so this optional step was skipped (a Hugging Face website "
          "needs a PRO account). The app link from step (a) works without it.")
else:
    # From the Colab secret HF_TOKEN if you made one, else pasted into the hidden box below
    # (a Write token from https://huggingface.co/settings/tokens; it is never shown or saved).
    hf_token = secret("HF_TOKEN") or getpass("Hugging Face Write token: ").strip()
    assert hf_token, "No token: create a Write token at https://huggingface.co/settings/tokens."
    git_ref = subprocess.run(["git", "-C", "/content/Skolfie", "rev-parse", "HEAD"],
                             capture_output=True, text=True, check=True).stdout.strip()

    run(["skolfie", "bundle", "--models-dir", MODELS_DIR, "--reports-dir", REPORTS_DIR,
         "--out", BUNDLE_DIR])
    build = ["skolfie", "space", "build", "--bundle", BUNDLE_DIR, "--out", SPACE_DIR,
             "--git-ref", git_ref]
    if PUBLIC_NO_LOGIN:
        build.append("--public")
    run(build)

    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub>=0.25"],
                   check=True)
    publish_env = dict(os.environ, HF_TOKEN=hf_token)
    if not PUBLIC_NO_LOGIN:
        publish_env["SKOLFIE_APP_AUTH"] = f"{APP_USER}:{APP_PASSWORD}"
    publish = ["skolfie", "space", "publish", "--dir", SPACE_DIR, "--repo-id", HF_SPACE]
    if SPACE_PRIVATE:
        publish.append("--private")
    run(publish, env=publish_env)

    print(f"\nBuild progress and logs: {space_page_url(HF_SPACE)}")
    print(f"The website (ready when the build finishes): {space_site_url(HF_SPACE)}")
    print("Log in with", APP_USER, "and your password." if not PUBLIC_NO_LOGIN else "(no login).")

Questions or odd results? Tell Claude in the project thread; send feedback from the app
for results that look wrong, so they can be used to improve the models.